# Ensemble Jaringan Saraf (MLP, LSTM, TCN) untuk Peramalan Keuangan Multi-Langkah

**Ide.** Data keuangan punya karakter berbeda-beda, dan setiap model peramalan secara teori hanya unggul pada kondisi tertentu. Model tunggal boleh kalah di kondisi yang tidak cocok dengannya. Karena itu model-model baseline dari **paradigma yang berbeda** digabung (ensemble), lalu diuji apakah gabungannya lebih kuat daripada masing-masing model tunggal, pada **8 data dengan karakter berbeda**.

**Yang dibandingkan**
- 3 model tunggal, **satu dari setiap paradigma jaringan saraf**: MLP (feedforward), LSTM (rekuren), TCN (konvolusional).
- 4 kombinasi = MLP+LSTM, MLP+TCN, LSTM+TCN, MLP+LSTM+TCN. Setiap kombinasi dihitung dengan **dua cara pembobotan**: **bobot setara** (rata-rata sederhana) dan **bobot Proposed** (bobot MSE-optimal yang diperbarui setiap 20 hari), sehingga ada 8 ensemble.

**Metrik:** RMSE, MAE, MAPE, MASE, DA.

**Dua versi dalam satu notebook** (atur `TUNING` di sel konfigurasi):
- `TUNING = False`: semua jaringan memakai setelan tetap yang sama (64 neuron, dropout 0,1, learning rate 0,001).
- `TUNING = True`: setiap model di-tuning ringan per aset dan per pipeline sebelum dilatih.

Hasil kedua versi disimpan terpisah. Jalankan notebook dua kali (sekali dengan `False`, sekali dengan `True`), dan bagian terakhir akan otomatis membandingkan keduanya.

**Cara pakai:** letakkan notebook ini satu folder dengan `data_keuangan.xlsx`, lalu **Run All**. Hasil training disimpan di `outputs/preds/`; kalau terputus, Run All lagi dan bagian yang sudah selesai dilewati. Untuk uji cepat set `QUICK = True` (hasilnya bukan untuk paper).

## 0. Instalasi & konfigurasi

In [ ]:
# Colab: hapus tanda '#' di baris berikut lalu jalankan
# !pip install -q statsmodels yfinance openpyxl optuna

In [ ]:
import os, time, json, pickle, warnings, copy, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
print("Device:", DEVICE)

In [ ]:
# ============================ KONFIGURASI ============================
QUICK = False                 # True = uji cepat (bukan untuk paper)
N_SEEDS = 5                   # jumlah pengulangan jaringan saraf dengan seed berbeda (3 atau 5)

ASSETS = ["SP500", "IHSG", "BTC", "GOLD", "BRENT", "TLT", "USDIDR", "EURUSD"]
DATA_FILE, OUT_DIR = "data_keuangan.xlsx", "outputs"
DATA_START, DATA_END = "2010-01-01", "2026-10-31"

HORIZONS = [1, 5, 10, 20]     # hari ke depan (BTC: hari kalender)
H_MAX = max(HORIZONS)
LOOKBACK = 60                 # panjang jendela input (hari)
TARGETS = ["return", "price"] # dua pipeline terpisah
TEST_YEARS = list(range(2020, 2027))
ES_DAYS = 250                 # blok early stopping
CAL_DAYS = 250                # blok kalibrasi: prediksi awal untuk menghitung bobot Proposed
STACK_WINDOW, REFIT_EVERY = 250, 20   # bobot Proposed: 250 prediksi terakhir yang sudah terealisasi, update tiap 20 hari

NN_MODELS = ["MLP", "LSTM", "TCN"]
EPOCHS, PATIENCE, BATCH = 100, 10, 64
TUNING = False                # False = setelan tetap; True = tuning ringan setiap model
HP = dict(HIDDEN=64, DROPOUT=0.1, LR=1e-3)   # setelan tetap (dipakai bila TUNING = False)
# tuning ringan: anggaran sama untuk setiap model, hanya memakai data sebelum tahun uji pertama
TUNE_TRIALS, TUNE_EPOCHS, TUNE_PATIENCE = 10, 30, 5
SEARCH_SPACE = dict(HIDDEN=[32, 64, 128], DROPOUT=(0.0, 0.3), LR=(1e-4, 3e-3))

SEEDS = list(range(N_SEEDS))
if QUICK:
    SEEDS, EPOCHS, PATIENCE, TEST_YEARS = [0, 1], 3, 2, [2020, 2025]
    TUNE_TRIALS, TUNE_EPOCHS, TUNE_PATIENCE = 2, 2, 1
os.makedirs(f"{OUT_DIR}/preds", exist_ok=True)
VERSION = "tuned" if TUNING else "v2"          # "v2" = nama file run sebelumnya (tanpa tuning), supaya bisa dipakai ulang
LABEL_VERSION = "dengan tuning" if TUNING else "tanpa tuning"
print("Versi yang dijalankan:", LABEL_VERSION)

## 1. Data dan karakteristiknya

Delapan data dipilih **berpasangan dan saling kontras**, supaya model diuji pada karakter data yang berbeda-beda:

| Pasangan | Data | Karakter yang dikontraskan |
|---|---|---|
| Saham | S&P 500 vs IHSG | Pasar maju vs pasar berkembang |
| Kurs | EUR/USD vs USD/IDR | Mengambang bebas vs dikelola bank sentral |
| Komoditas | Emas vs Brent | Safe haven bertren vs energi dengan guncangan pasokan |
| Lainnya | Bitcoin vs TLT (obligasi AS) | Volatilitas ekstrem vs obligasi berekor ringan |

Sumber: Yahoo Finance; kurs dari kurs referensi ECB. Fitur input: deret itu sendiri + log VIX (indikator ketakutan pasar).

In [ ]:
def remove_fx_spikes(close, thr=0.03, rev=0.25):
    # buang kuotasi kurs yang melonjak >3% lalu berbalik penuh esok harinya (salah input)
    r = np.log(close).diff(); nxt = r.shift(-1)
    bad = (r.abs() > thr) & ((r + nxt).abs() < rev * r.abs())
    return close[~bad]

def download_series(tk):
    if tk.startswith("ECB:"):
        url = ("https://data-api.ecb.europa.eu/service/data/EXR/D.IDR+USD.EUR.SP00.A"
               f"?format=csvdata&startPeriod={DATA_START}")
        e = pd.read_csv(url, usecols=["CURRENCY", "TIME_PERIOD", "OBS_VALUE"])
        p = e.pivot(index="TIME_PERIOD", columns="CURRENCY", values="OBS_VALUE"); p.index = pd.to_datetime(p.index)
        s = p["USD"].dropna() if tk.endswith("EURUSD") else (p["IDR"] / p["USD"]).dropna()
        s = remove_fx_spikes(s.sort_index())
    else:
        import yfinance as yf
        d = yf.download(tk, start=DATA_START, end=DATA_END, progress=False, auto_adjust=True)
        if isinstance(d.columns, pd.MultiIndex): d.columns = [c[0] for c in d.columns]
        s = d["Close"].dropna()
    s = s[s.index < DATA_END]
    return pd.DataFrame({"Close": s.values}, index=pd.Index(s.index.date, name="Date"))

TICKERS = {"SP500": "^GSPC", "IHSG": "^JKSE", "BTC": "BTC-USD", "GOLD": "GC=F", "BRENT": "BZ=F", "TLT": "TLT",
           "USDIDR": "ECB:USDIDR", "EURUSD": "ECB:EURUSD", "VIX": "^VIX"}
if not os.path.exists(DATA_FILE):
    print("File data belum ada, mengunduh ...")
    with pd.ExcelWriter(DATA_FILE) as xw:
        for name, tk in TICKERS.items():
            download_series(tk).to_excel(xw, sheet_name=name)
RAW = pd.read_excel(DATA_FILE, sheet_name=None, index_col=0, parse_dates=True)

def load_asset(name):
    df = pd.DataFrame({"close": RAW[name]["Close"].astype(float)})
    df = df[~df.index.duplicated()].sort_index()
    df = df[(df.index >= DATA_START) & (df.index < DATA_END)]
    df["lvix"] = np.log(RAW["VIX"]["Close"].reindex(df.index).ffill())   # tanpa back-fill (tidak memakai masa depan)
    df = df.dropna(subset=["lvix"])
    df["r"] = np.log(df["close"]).diff()
    return df

DATA = {a: load_asset(a) for a in ASSETS}
pd.DataFrame({a: [d.index[0].date(), d.index[-1].date(), len(d)] for a, d in DATA.items()},
             index=["Mulai", "Akhir", "N"]).T

### Tabel 1. Karakteristik data
| Kolom | Arti |
|---|---|
| Vol (%) | Volatilitas tahunan |
| Drift/Vol | Kekuatan tren |
| Skew, ExKurt | Kemiringan dan ekor tebal |
| LB(10) p | Uji Ljung-Box: p < 0,05 = ada autokorelasi linier |
| ARCH(10) | Volatilitas bergerombol (makin besar makin kuat) |
| VR(20) | Variance ratio: > 1 momentum, < 1 mean reversion |
| RegimeRatio | Volatilitas maksimum ÷ median: makin besar, makin sering berganti rezim |

In [ ]:
def characteristics(close):
    from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
    from scipy import stats
    r = np.log(close).diff().dropna()
    ann = len(r) / ((r.index[-1] - r.index[0]).days / 365.25)
    rv = r.rolling(60).std().dropna()
    return {"Vol (%)": 100 * r.std() * np.sqrt(ann), "Drift/Vol": r.mean() * ann / (r.std() * np.sqrt(ann)),
            "Skew": stats.skew(r), "ExKurt": stats.kurtosis(r),
            "LB(10) p": acorr_ljungbox(r, lags=[10])["lb_pvalue"].iloc[0],
            "ARCH(10)": het_arch(r, nlags=10)[0],
            "VR(20)": np.log(close).diff(20).dropna().var() / (20 * r.var()),
            "RegimeRatio": rv.max() / rv.median()}

TABLE1 = pd.DataFrame({a: characteristics(d["close"]) for a, d in DATA.items()}).T.round(3)
TABLE1

## 2. Model dan alasan pemilihannya

**Satu model untuk setiap paradigma jaringan saraf**, karena keberagaman cara membaca data itulah yang membuat ensemble berguna. Kalau semua model berasumsi sama, kesalahan mereka juga cenderung sama.

| Model | Paradigma | Asumsi / kekuatan teoretis | Secara teori unggul pada | Referensi |
|---|---|---|---|---|
| **MLP** | Jaringan saraf feedforward | Pendekatan universal untuk hubungan nonlinier antara 60 hari terakhir dan masa depan, tanpa memperhatikan urutan | Pola nonlinier yang tidak bergantung urutan waktu | Hornik et al. (1989) |
| **LSTM** | Jaringan saraf rekuren | Membaca data berurutan dengan memori jangka panjang (gerbang ingat/lupa) | Pola yang bertahan lama, tren, dan rezim, mis. **BTC, USD/IDR** (VR > 1) | Hochreiter & Schmidhuber (1997) |
| **TCN** | Jaringan saraf konvolusional | Filter konvolusi bergeser di sepanjang sumbu waktu dan mengenali *bentuk* pola lokal; dilatasi 1-2-4-8-16 menangkap beberapa skala waktu sekaligus | Pola lokal dan guncangan, mis. **Brent, BTC** | Bai, Kolter & Koltun (2018) |

**Kenapa GRU tidak dipakai?** GRU satu paradigma dengan LSTM (rekuren), sehingga tidak menambah keberagaman cara membaca data.

**Seed.** Hasil jaringan saraf bisa berubah tergantung titik awal acak (seed). Setiap jaringan dilatih **N_SEEDS kali** dengan seed berbeda, dan setiap kombinasi ensemble dibentuk **per seed** (misalnya MLP+LSTM seed 0 menggabungkan MLP seed 0 dan LSTM seed 0). Hasil ditulis rata-rata ± simpangan baku antar seed, sehingga terlihat apakah keunggulan ensemble konsisten dan tidak bergantung keberuntungan seed.

In [ ]:
# ---------------- jaringan saraf: satu jaringan mengeluarkan prediksi untuk semua horizon sekaligus ----------------
class MLPNet(nn.Module):
    def __init__(s, n_feat, n_out, hid, drop):
        super().__init__()
        s.net = nn.Sequential(nn.Flatten(), nn.Linear(LOOKBACK * n_feat, 2 * hid), nn.ReLU(), nn.Dropout(drop),
                              nn.Linear(2 * hid, hid), nn.ReLU(), nn.Dropout(drop), nn.Linear(hid, n_out))
    def forward(s, x): return s.net(x)

class LSTMNet(nn.Module):
    def __init__(s, n_feat, n_out, hid, drop):
        super().__init__()
        s.rnn, s.drop, s.head = nn.LSTM(n_feat, hid, batch_first=True), nn.Dropout(drop), nn.Linear(hid, n_out)
    def forward(s, x): return s.head(s.drop(s.rnn(x)[0][:, -1]))

class TCNBlock(nn.Module):
    def __init__(s, cin, cout, k, d, drop):
        super().__init__()
        s.pad, s.conv, s.drop = (k - 1) * d, nn.Conv1d(cin, cout, k, dilation=d), nn.Dropout(drop)
        s.res = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(s, x):     # padding kiri saja -> kausal (tidak melihat masa depan)
        return F.relu(s.drop(F.relu(s.conv(F.pad(x, (s.pad, 0))))) + s.res(x))

class TCNNet(nn.Module):
    def __init__(s, n_feat, n_out, hid, drop):
        super().__init__()
        blocks, c = [], n_feat
        for d in (1, 2, 4, 8, 16):                   # receptive field 63 hari >= LOOKBACK
            blocks.append(TCNBlock(c, hid, 3, d, drop)); c = hid
        s.blocks, s.head = nn.Sequential(*blocks), nn.Linear(hid, n_out)
    def forward(s, x): return s.head(s.blocks(x.transpose(1, 2))[:, :, -1])

NETS = {"MLP": MLPNet, "LSTM": LSTMNet, "TCN": TCNNet}

def train_net(name, seed, Xtr, Ytr, Xes, Yes, hp=None, epochs=None, patience=None):
    hp, epochs, patience = hp or HP, epochs or EPOCHS, patience or PATIENCE
    torch.manual_seed(seed); np.random.seed(seed)
    net = NETS[name](Xtr.shape[-1], Ytr.shape[-1], hp["HIDDEN"], hp["DROPOUT"]).to(DEVICE)
    opt, loss_fn = torch.optim.Adam(net.parameters(), lr=hp["LR"]), nn.MSELoss()
    Xtr, Ytr, Xes, Yes = [torch.from_numpy(a).to(DEVICE) for a in (Xtr, Ytr, Xes, Yes)]
    g = torch.Generator().manual_seed(seed)
    best, best_state, wait = np.inf, copy.deepcopy(net.state_dict()), 0
    for _ in range(epochs):
        net.train()
        perm = torch.randperm(len(Xtr), generator=g).to(DEVICE)
        for b in range(0, len(perm), BATCH):
            i = perm[b:b + BATCH]
            loss = loss_fn(net(Xtr[i]), Ytr[i])
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        net.eval()
        with torch.no_grad(): v = loss_fn(net(Xes), Yes).item()
        if v < best - 1e-7: best, best_state, wait = v, copy.deepcopy(net.state_dict()), 0
        else:
            wait += 1
            if wait >= patience: break
    net.load_state_dict(best_state); net.eval()
    net.best_loss = best
    return net

@torch.no_grad()
def predict_net(net, X):
    return net(torch.from_numpy(X).to(DEVICE)).cpu().numpy().astype(float)

## 3. Desain eksperimen

**Dua pipeline terpisah**
- **Return**: target = return log kumulatif r(t→t+h). Input = return harian 60 hari terakhir + log VIX. Dinormalisasi dengan standardisasi.
- **Harga**: target = harga P(t+h). Input = harga 60 hari terakhir + log VIX. Dinormalisasi dengan **min-max**.

Semua normalisasi **di-fit hanya di data latih**. Prediksi multi-langkah dibuat langsung (direct/MIMO): satu jaringan mengeluarkan prediksi untuk h = 1, 5, 10, 20 sekaligus.

**Walk-forward per tahun uji (2020–2026):** `latih | jeda | early stopping (250 hari) | jeda | kalibrasi (250 hari) | jeda | uji (1 tahun)`.
- **Latih**: parameter model dan normalisasi.
- **Early stopping**: kapan jaringan saraf berhenti dilatih.
- **Kalibrasi**: prediksi di sini dipakai sebagai riwayat awal untuk menghitung bobot Proposed di hari-hari pertama tahun uji. Tidak pernah dipakai untuk melatih model.
- Model dilatih ulang setiap tahun. Jeda 20 hari memastikan target suatu blok sudah terjadi sebelum blok berikutnya dimulai, jadi **tidak ada kebocoran data**.

In [ ]:
def build_samples(df, target):
    close, r, lvix = df["close"].values, df["r"].fillna(0).values, df["lvix"].values
    n, L = len(df), LOOKBACK
    pos = np.arange(L, n)
    base = r if target == "return" else close
    X = np.stack([np.stack([base[i-L+1:i+1], lvix[i-L+1:i+1]], -1) for i in pos])
    Y = np.full((len(pos), len(HORIZONS)), np.nan)
    for k, h in enumerate(HORIZONS):
        ok = pos + h < n
        Y[ok, k] = np.log(close[pos[ok]+h] / close[pos[ok]]) if target == "return" else close[pos[ok]+h]
    return dict(X=X, Y=Y, pos=pos, dates=df.index[pos], p0=close[pos], close=close, r=df["r"].values, index=df.index)

def make_fold(S, year):
    pos, dates = S["pos"], S["dates"]
    tm = dates.year == year
    if not tm.any(): return None
    cal = np.where(pos + H_MAX < pos[tm][0])[0][-CAL_DAYS:]
    es = np.where(pos + H_MAX < pos[cal[0]])[0][-ES_DAYS:]
    tr = np.where(pos + H_MAX < pos[es[0]])[0]
    return dict(train=tr, es=es, cal=cal, test=np.where(tm)[0])

class Scaler:
    # di-fit HANYA di blok latih. return: standardisasi; harga: min-max. (VIX: standardisasi)
    def __init__(s, S, fold, target):
        tr = fold["train"]; X, Y = S["X"][tr], S["Y"][tr]
        s.target, s.vm, s.vs = target, X[..., 1].mean(), X[..., 1].std()
        if target == "return":
            s.xm, s.xs, s.ym, s.ys = X[..., 0].mean(), X[..., 0].std(), Y.mean(0), Y.std(0)
        else:
            seen = S["close"][: S["pos"][tr[-1]] + H_MAX + 1]    # semua harga yang terlihat di blok latih
            s.lo, s.hi = seen.min(), seen.max()
    def x(s, X):
        Z = np.empty_like(X, dtype=np.float32)
        Z[..., 1] = (X[..., 1] - s.vm) / s.vs
        Z[..., 0] = (X[..., 0] - s.xm) / s.xs if s.target == "return" else (X[..., 0] - s.lo) / (s.hi - s.lo)
        return Z
    def y(s, Y):
        return ((Y - s.ym) / s.ys if s.target == "return" else (Y - s.lo) / (s.hi - s.lo)).astype(np.float32)
    def inv(s, Z):
        return Z * s.ys + s.ym if s.target == "return" else Z * (s.hi - s.lo) + s.lo

# cek: tidak ada label yang tumpang tindih antar blok
S_ = build_samples(DATA["IHSG"], "return"); f_ = make_fold(S_, 2022)
for b1, b2 in (("train", "es"), ("es", "cal"), ("cal", "test")):
    assert S_["index"][S_["pos"][f_[b1][-1]] + H_MAX] < S_["dates"][f_[b2][0]]
print("Contoh fold IHSG 2022:", {b: f"{S_['dates'][f_[b][0]].date()} s.d. {S_['dates'][f_[b][-1]].date()}" for b in f_})

### Tuning ringan (hanya bila `TUNING = True`)
Setiap model (MLP, LSTM, TCN) diberi **anggaran yang sama**: 10 percobaan Optuna (TPE) per aset dan per pipeline, masing-masing maksimal 30 epoch.
- Ruang pencarian: jumlah neuron {32, 64, 128}, dropout 0–0,3, learning rate 10⁻⁴–3·10⁻³.
- Tuning **hanya memakai blok latih dan early stopping pada fold pertama** (semua sebelum 2020), sehingga data uji tidak pernah disentuh.
- Setelan terbaik dibekukan untuk semua tahun dan semua seed.

Kalau `TUNING = False`, semua model memakai setelan tetap `HP`.

In [ ]:
HP_FILE = f"{OUT_DIR}/hparams_{VERSION}.json"
if TUNING:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HPARAMS = json.load(open(HP_FILE)) if os.path.exists(HP_FILE) else {}
    for asset in ASSETS:
        for target in TARGETS:
            S = build_samples(DATA[asset], target); fold = make_fold(S, TEST_YEARS[0]); sc = Scaler(S, fold, target)
            Xtr, Ytr = sc.x(S["X"][fold["train"]]), sc.y(S["Y"][fold["train"]])
            Xes, Yes = sc.x(S["X"][fold["es"]]), sc.y(S["Y"][fold["es"]])
            for name in NN_MODELS:
                key = f"{asset}|{target}|{name}"
                if key in HPARAMS: continue
                t0 = time.time()
                def objective(trial):
                    hp = dict(HIDDEN=trial.suggest_categorical("HIDDEN", SEARCH_SPACE["HIDDEN"]),
                              DROPOUT=trial.suggest_float("DROPOUT", *SEARCH_SPACE["DROPOUT"]),
                              LR=trial.suggest_float("LR", *SEARCH_SPACE["LR"], log=True))
                    return train_net(name, 0, Xtr, Ytr, Xes, Yes, hp, TUNE_EPOCHS, TUNE_PATIENCE).best_loss
                study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0))
                study.optimize(objective, n_trials=TUNE_TRIALS)
                HPARAMS[key] = study.best_params
                json.dump(HPARAMS, open(HP_FILE, "w"), indent=1)
                print(f"{asset:7s} {target:6s} {name:4s} {study.best_params}  ({time.time()-t0:.0f} dtk)")
else:
    HPARAMS = {f"{a}|{t}|{m}": HP for a in ASSETS for t in TARGETS for m in NN_MODELS}
HP_TABLE = pd.DataFrame({tuple(k.split("|")): v for k, v in HPARAMS.items()}).T
HP_TABLE.index.names = ["Aset", "Pipeline", "Model"]
display(Markdown(f"**Hyperparameter yang dipakai ({LABEL_VERSION})**")); HP_TABLE.round(4)

In [ ]:
# ---------------- jalankan semua model, simpan prediksi per fold (bisa dilanjutkan bila terputus) ----------------
def run_fold(asset, target, S, year):
    fold = make_fold(S, year)
    if fold is None: return None
    sc = Scaler(S, fold, target)
    tr, es, te = fold["train"], fold["es"], fold["test"]
    ix = np.concatenate([fold["cal"], te])            # prediksi dibuat untuk blok kalibrasi + uji
    Xtr, Ytr, Xes, Yes, Xix = sc.x(S["X"][tr]), sc.y(S["Y"][tr]), sc.x(S["X"][es]), sc.y(S["Y"][es]), sc.x(S["X"][ix])
    pred = {}
    for name in NN_MODELS:
        hp = HPARAMS[f"{asset}|{target}|{name}"]
        pred[name] = np.stack([sc.inv(predict_net(train_net(name, s, Xtr, Ytr, Xes, Yes, hp), Xix)) for s in SEEDS])
    if target == "return": scale = np.nanmean(np.abs(S["Y"][tr]), 0)
    else:                  scale = np.nanmean(np.abs(S["Y"][tr] - S["p0"][tr, None]), 0)
    nc = len(fold["cal"])
    return dict(pred=pred, n_cal=nc, y_all=S["Y"][ix], pos_all=S["pos"][ix], p0_all=S["p0"][ix],
                y=S["Y"][te], p0=S["p0"][te], dates=S["dates"][te], mase_scale=scale)

jobs = [(a, t, y) for a in ASSETS for t in TARGETS for y in TEST_YEARS]
t0, done = time.time(), 0
for asset, target, year in jobs:
    path = f"{OUT_DIR}/preds/{asset}_{target}_{year}_s{len(SEEDS)}_{VERSION}.pkl"
    if os.path.exists(path): continue
    t1 = time.time(); out = run_fold(asset, target, build_samples(DATA[asset], target), year)
    if out is not None: pickle.dump(out, open(path, "wb"))
    done += 1
    left = sum(not os.path.exists(f"{OUT_DIR}/preds/{a}_{t}_{y}_s{len(SEEDS)}_{VERSION}.pkl") for a, t, y in jobs)
    print(f"{asset:7s} {target:6s} {year}  {time.time()-t1:5.0f} dtk | sisa {left} fold, ~{(time.time()-t0)/done*left/60:.0f} menit")
print("Semua fold selesai.")

## 4. Ensemble

Semua kombinasi **2 dan 3 model** dari {MLP, LSTM, TCN} (4 kombinasi). Setiap kombinasi dihitung dengan dua cara pembobotan:

| Pembobotan | Cara kerja | Alasan |
|---|---|---|
| **Setara** | Rata-rata sederhana, setiap model bersuara sama | Pembanding klasik yang sering sulit dikalahkan (*forecast combination puzzle*; Clemen, 1989; Stock & Watson, 2004) dan tanpa parameter tambahan |
| **Proposed** | Bobot w ≥ 0, Σw = 1 yang meminimalkan error kuadrat gabungan pada **250 prediksi terakhir yang targetnya sudah terjadi**, diperbarui setiap 20 hari (Granger & Ramanathan, 1984) | Bobot otomatis bergeser ke model yang sedang cocok dengan karakter data, sehingga model yang sedang lemah tidak menyeret hasil gabungan |

Bobot Proposed **tidak pernah memakai informasi masa depan**: di hari t, hanya prediksi yang targetnya sudah terealisasi sebelum t yang dipakai. Di awal tahun uji, riwayatnya diambil dari blok kalibrasi.

Ensemble dibentuk **per seed**: misalnya MLP+LSTM untuk seed 0 menggabungkan MLP dan LSTM seed 0.

In [ ]:
from scipy.optimize import minimize

def mse_weights(y, F):
    # bobot di simpleks yang meminimalkan mean (y - w'F)^2 ;  F: (M, N)
    ok = ~np.isnan(y); y, F = y[ok], F[:, ok]
    Q, c = F @ F.T / len(y), F @ y / len(y)
    sc = np.abs(np.diag(Q)).mean(); Q, c = Q / sc, c / sc
    M = len(c)
    res = minimize(lambda w: w @ Q @ w - 2 * c @ w, np.full(M, 1 / M), jac=lambda w: 2 * Q @ w - 2 * c,
                   method="SLSQP", bounds=[(0, 1)] * M,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(M)}],
                   options={"ftol": 1e-14, "maxiter": 1000})
    w = np.clip(res.x, 0, None); return w / w.sum()

def proposed_weights(F_all, y_all, pos_all, p0_all, n_cal, target):
    # F_all: (M, N_cal+N_test, H). Bobot untuk setiap origin uji, (M, N_test, H)
    if target == "price":                     # pusatkan pada harga hari ini: solusi sama, numerik lebih stabil
        F_all, y_all = F_all - p0_all[None, :, None], y_all - p0_all[:, None]
    M, N, H = F_all.shape; Nt = N - n_cal
    W = np.empty((M, Nt, len(HORIZONS)))
    for k, h in enumerate(HORIZONS):
        for s in range(0, Nt, REFIT_EVERY):
            i = n_cal + s
            avail = np.where((pos_all + h <= pos_all[i]) & ~np.isnan(y_all[:, k]))[0][-STACK_WINDOW:]
            W[:, s:s + REFIT_EVERY, k] = mse_weights(y_all[avail, k], F_all[:, avail, k])[:, None]
    return W

SINGLES = NN_MODELS
COMBOS = [c for k in range(2, len(SINGLES) + 1) for c in itertools.combinations(SINGLES, k)]
SCHEMES = {"setara": "Setara", "proposed": "Proposed"}
CNAME = {(c, sc): f"{'+'.join(c)} ({SCHEMES[sc]})" for c in COMBOS for sc in SCHEMES}
print(len(CNAME), "ensemble")

def as_seeds(p):     # berjaga-jaga bila ada prediksi tanpa dimensi seed
    return p if p.ndim == 3 else np.repeat(p[None], len(SEEDS), 0)

def metrics(y, f, p0, scale, target):
    # scale: penyebut MASE per origin (MAE tebakan "tidak berubah" di blok latih fold-nya)
    ok = ~np.isnan(y); y, f, p0, scale = y[ok], f[ok], p0[ok], scale[ok]
    e = y - f
    ref = np.zeros_like(y) if target == "return" else p0
    dy, df_ = y - ref, f - ref; nz = (dy != 0) & (np.abs(df_) > 1e-12)
    return dict(RMSE=np.sqrt(np.mean(e ** 2)), MAE=np.mean(np.abs(e)),
                MAPE=100 * np.mean(np.abs(e / y)) if target == "price" else np.nan,
                MASE=np.mean(np.abs(e) / scale),
                DA=100 * np.mean(np.sign(df_[nz]) == np.sign(dy[nz])) if nz.any() else np.nan)

rows, WROWS = [], []
for asset in ASSETS:
    for target in TARGETS:
        folds = [pickle.load(open(f, "rb")) for f in
                 [f"{OUT_DIR}/preds/{asset}_{target}_{y}_s{len(SEEDS)}_{VERSION}.pkl" for y in TEST_YEARS] if os.path.exists(f)]
        if not folds: continue
        P = {m: [] for m in SINGLES + list(CNAME.values())}
        for d in folds:
            nc = d["n_cal"]
            full = {m: as_seeds(d["pred"][m]) for m in SINGLES}                     # (seed, N_cal+N_test, H)
            for m in SINGLES: P[m].append(full[m][:, nc:])
            for c in COMBOS:
                F = np.stack([full[m] for m in c], 1)                             # (seed, M, N, H)
                P[CNAME[(c, "setara")]].append(F[:, :, nc:].mean(1))
                ens = []
                for s in range(len(SEEDS)):
                    W = proposed_weights(F[s], d["y_all"], d["pos_all"], d["p0_all"], nc, target)
                    ens.append((W * F[s, :, nc:]).sum(0))
                    if len(c) == len(SINGLES):                                     # bobot kombinasi 3 model, untuk dilaporkan
                        for k, h in enumerate(HORIZONS):
                            WROWS.append(dict(asset=asset, target=target, h=h, seed=s, **{m: W[j, :, k].mean() for j, m in enumerate(c)}))
                P[CNAME[(c, "proposed")]].append(np.stack(ens))
        P = {m: np.concatenate(v, 1) for m, v in P.items()}
        y = np.concatenate([d["y"] for d in folds]); p0 = np.concatenate([d["p0"] for d in folds])
        scale = np.concatenate([np.tile(d["mase_scale"], (len(d["y"]), 1)) for d in folds])
        for model, pr in P.items():
            for k, h in enumerate(HORIZONS):
                for s in range(len(SEEDS)):
                    rows.append(dict(asset=asset, target=target, model=model, h=h, seed=s,
                                     **metrics(y[:, k], pr[s, :, k], p0, scale[:, k], target)))
        print(f"selesai {asset} {target}")
RES = pd.DataFrame(rows)
WEIGHTS = pd.DataFrame(WROWS)
ENS = list(CNAME.values())
MODELS = SINGLES + ENS
TYPE = {m: "Tunggal" if m in SINGLES else
        f"Ensemble {m.split(' (')[0].count('+') + 1} model, {m.split('(')[1][:-1]}" for m in MODELS}

## 5. Hasil

### 5.1 Ringkasan: model tunggal vs ensemble
Rata-rata semua aset dan horizon (± simpangan baku antar seed). **Peringkat** = rata-rata peringkat di 32 sel (8 aset × 4 horizon), 1 = terbaik. MAPE hanya untuk pipeline harga (return bisa bernilai nol, sehingga MAPE tidak terdefinisi). Karena RMSE dan MAE punya satuan berbeda antar aset, perbandingan lintas aset memakai **peringkat**. MASE bebas satuan (< 1 = lebih baik dari tebakan "tidak berubah").

In [ ]:
def summary(target):
    r = RES[RES.target == target]
    cell = r.groupby(["model", "asset", "h"])[["RMSE", "MAE", "MAPE", "MASE", "DA"]].mean()      # rata-rata seed
    out = {}
    for met in ["RMSE", "MAE", "MAPE", "MASE", "DA"]:
        if target == "return" and met == "MAPE": continue
        m = cell[met].unstack(["asset", "h"]).reindex(MODELS)
        better = "max" if met == "DA" else "min"
        out[f"Peringkat {met}"] = (m.rank(ascending=(better == "min")).mean(1))
    seed_sd = r.groupby(["model", "seed"])["MASE"].mean().groupby("model").std().reindex(MODELS)
    out["MASE rata-rata"] = cell["MASE"].groupby("model").mean().reindex(MODELS)
    out["sd antar seed (MASE)"] = seed_sd
    df = pd.DataFrame(out); df.insert(0, "Jenis", [TYPE[m] for m in df.index])
    return df.round(4)

for t in TARGETS:
    display(Markdown(f"**Pipeline {t}**"))
    s = summary(t)
    display(s.style.highlight_min(subset=[c for c in s.columns if c.startswith("Peringkat") and "DA" not in c], color="#fff2cc")
                   .format(precision=4))

### 5.2 Apakah ensemble memperkuat model tunggalnya?
Untuk setiap ensemble (bobot setara dan bobot Proposed): dibandingkan dengan **model terbaik di dalam kombinasinya** dan dengan **rata-rata anggotanya**, di setiap aset × horizon. Kolom "menang" = berapa dari 32 sel ensemble lebih baik dari anggota terbaiknya. Ini menjawab langsung pertanyaan "apakah menggabungkan model memperkuat baseline?".

In [ ]:
def strengthen(target, met):
    cell = RES[RES.target == target].groupby(["model", "asset", "h"])[met].mean().unstack(["asset", "h"])
    rows = []
    for (c, sc), name in CNAME.items():
        e, mem = cell.loc[name], cell.loc[list(c)]
        best, avg = mem.min(), mem.mean()
        rows.append({"Ensemble": name, "Jumlah model": len(c), "Bobot": SCHEMES[sc],
                     f"Perbaikan vs anggota terbaik (%)": (100 * (best - e) / best).mean(),
                     f"Perbaikan vs rata-rata anggota (%)": (100 * (avg - e) / avg).mean(),
                     "Menang vs anggota terbaik": f"{int((e < best).sum())}/{len(e)}",
                     "Menang vs semua model tunggal": f"{int((e < cell.loc[SINGLES].min()).sum())}/{len(e)}"})
    return pd.DataFrame(rows).round(2)

for t, met in [("return", "RMSE"), ("price", "RMSE"), ("price", "MAPE")]:
    display(Markdown(f"**{t} — {met}**")); display(strengthen(t, met))

### 5.3 Detail per aset × horizon
Sel tebal kuning = terbaik di kolom. Jaringan saraf dan ensemble = rata-rata antar seed.

In [ ]:
def detail(target, met, digits=4):
    m = RES[RES.target == target].groupby(["model", "asset", "h"])[met].mean().unstack(["asset", "h"]).reindex(MODELS)
    m = m[[c for a in ASSETS for c in m.columns if c[0] == a]]
    best = m.max() if met == "DA" else m.min()
    sty = m.style.format(f"{{:.{digits}g}}").apply(lambda col: ["font-weight:bold; background:#fff2cc" if v == best[col.name] else "" for v in col])
    display(sty)
    return m

DETAIL = {}
for t, met in [("return", "RMSE"), ("return", "MAE"), ("price", "RMSE"), ("price", "MAE"), ("price", "MAPE")]:
    display(Markdown(f"**{t} — {met}**"))
    DETAIL[(t, met)] = detail(t, met)

### 5.4 Model tunggal terbaik berganti-ganti antar data
Kalau model tunggal terbaiknya berbeda untuk setiap aset dan horizon, artinya tidak ada satu model yang selalu terbaik. Inilah alasan utama memakai ensemble.

In [ ]:
for t, met in [("return", "RMSE"), ("price", "RMSE"), ("price", "MAPE")]:
    m = DETAIL[(t, met)].loc[SINGLES]
    display(Markdown(f"**{t} — {met}: model tunggal terbaik**"))
    display(m.idxmin().unstack("h").reindex(ASSETS))

### 5.5 Uji statistik: Bayesian signed-rank (Benavoli et al., 2017)
Membandingkan setiap ensemble dengan setiap model tunggal **lintas 32 sel** (8 aset × 4 horizon), dan juga bobot Proposed vs bobot setara untuk kombinasi yang sama (kolom "versi Setara"). Selisih RMSE/MAPE di bawah 1% dianggap setara (ROPE). Hasilnya berupa peluang: ensemble lebih baik, setara, atau lebih buruk.

In [ ]:
ROPE = 0.01
def bayes_signed_rank(z, rope=ROPE, n_samples=20000, seed=0):
    rng = np.random.default_rng(seed)
    z = np.concatenate([[0.0], np.asarray(z, float)])           # pengamatan prior di 0
    W = rng.dirichlet(np.ones(len(z)), n_samples)
    S = z[:, None] + z[None, :]
    th = np.stack([np.einsum("ni,ij,nj->n", W, M.astype(float), W)
                   for M in (S < -2 * rope, np.abs(S) <= 2 * rope, S > 2 * rope)], 1)
    p = np.bincount(th.argmax(1), minlength=3) / n_samples
    return {"P(ensemble lebih buruk)": p[0], "P(setara)": p[1], "P(ensemble lebih baik)": p[2]}

rows = []
for t, met in [("return", "RMSE"), ("price", "RMSE"), ("price", "MAPE")]:
    m = DETAIL[(t, met)]
    for name in ENS + []:
        for s_ in SINGLES:
            z = (m.loc[s_] - m.loc[name]) / m.loc[s_]                  # positif = ensemble lebih baik
            rows.append({"Pipeline": f"{t} {met}", "Ensemble": name, "vs model tunggal": s_, **bayes_signed_rank(z.values)})
    for c in COMBOS:                                                     # Proposed vs Setara (kombinasi yang sama)
        z = (m.loc[CNAME[(c, "setara")]] - m.loc[CNAME[(c, "proposed")]]) / m.loc[CNAME[(c, "setara")]]
        rows.append({"Pipeline": f"{t} {met}", "Ensemble": CNAME[(c, "proposed")], "vs model tunggal": "versi Setara", **bayes_signed_rank(z.values)})
BAYES = pd.DataFrame(rows).round(3)
display(BAYES.pivot_table(index=["Pipeline", "Ensemble"], columns="vs model tunggal", values="P(ensemble lebih baik)", sort=False)
        .style.background_gradient(cmap="Greens", vmin=0, vmax=1).format("{:.2f}"))

### 5.6 Bobot Proposed: model mana yang dipercaya untuk tiap data?
Rata-rata porsi bobot pada kombinasi 3 model (rata-rata antar seed dan sepanjang periode uji). Bandingkan dengan Tabel 1 untuk melihat karakter data apa yang membuat ensemble lebih mempercayai MLP, LSTM, atau TCN.

In [ ]:
WTAB = WEIGHTS.groupby(["target", "asset", "h"])[SINGLES].mean().round(3)
for t in TARGETS:
    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(18, 4), sharey=True)
    for ax, h in zip(axes, HORIZONS):
        w = WTAB.loc[t].xs(h, level="h").reindex(ASSETS)
        w.plot.bar(stacked=True, ax=ax, legend=False, width=0.8, color=["#ffb74d", "#ef5350", "#8d6e63"])
        ax.set_title(f"{t}, h = {h}"); ax.set_xlabel(""); ax.set_ylim(0, 1)
    axes[0].set_ylabel("Porsi bobot"); fig.legend(SINGLES, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.08))
    plt.tight_layout(); plt.show()
WTAB

### 5.7 Grafik: posisi ensemble dibanding model tunggal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax, t in zip(axes, TARGETS):
    s = summary(t)["Peringkat RMSE"].sort_values()
    colors = ["#5c6bc0" if TYPE[m] == "Tunggal"
              else "#ef6c00" if "Proposed" in m else "#ffcc80" for m in s.index]
    ax.barh(s.index[::-1], s.values[::-1], color=colors[::-1])
    ax.set_title(f"Pipeline {t}: rata-rata peringkat RMSE (kecil = lebih baik)"); ax.set_xlabel("Peringkat")
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color="#5c6bc0", label="Model tunggal"), Patch(color="#ffcc80", label="Ensemble bobot setara"),
                    Patch(color="#ef6c00", label="Ensemble bobot Proposed")],
           loc="upper center", ncol=3)
plt.tight_layout(rect=(0, 0, 1, 0.93)); plt.show()

### 5.8 Simpan semua tabel ke Excel

In [ ]:
RES.to_pickle(f"{OUT_DIR}/RES_{VERSION}.pkl")          # dipakai untuk membandingkan versi tanpa vs dengan tuning
XLS = f"{OUT_DIR}/hasil_ensemble_nn_{'tuning' if TUNING else 'tanpa_tuning'}.xlsx"
with pd.ExcelWriter(XLS) as xw:
    TABLE1.to_excel(xw, sheet_name="karakteristik_data")
    HP_TABLE.to_excel(xw, sheet_name="hyperparameter")
    for t in TARGETS: summary(t).to_excel(xw, sheet_name=f"ringkasan_{t}")
    for t, met in [("return", "RMSE"), ("price", "RMSE"), ("price", "MAPE")]:
        strengthen(t, met).to_excel(xw, sheet_name=f"perkuat_{t}_{met}", index=False)
    for (t, met), m in DETAIL.items(): m.to_excel(xw, sheet_name=f"detail_{t}_{met}")
    BAYES.to_excel(xw, sheet_name="bayes_signed_rank", index=False)
    WTAB.to_excel(xw, sheet_name="bobot_proposed_4model")
    RES.to_excel(xw, sheet_name="semua_metrik_per_seed", index=False)
print("Tersimpan:", XLS)

### 5.9 Perbandingan versi tanpa tuning vs dengan tuning
Bagian ini hanya tampil kalau notebook sudah dijalankan **dua kali** (sekali `TUNING = False`, sekali `TUNING = True`).
- Tabel pertama: MAPE dan RMSE rata-rata setiap model pada kedua versi. Perubahan (%) positif = tuning memperbaiki.
- Tabel kedua: apakah keunggulan ensemble atas anggota terbaiknya tetap ada setelah model tunggal di-tuning. Ini yang biasanya ditanyakan reviewer.

In [ ]:
f0, f1 = f"{OUT_DIR}/RES_v2.pkl", f"{OUT_DIR}/RES_tuned.pkl"
if os.path.exists(f0) and os.path.exists(f1):
    R0, R1 = pd.read_pickle(f0), pd.read_pickle(f1)
    rows = []
    for t, met in [("return", "RMSE"), ("price", "RMSE"), ("price", "MAPE")]:
        c0 = R0[R0.target == t].groupby(["model", "asset", "h"])[met].mean().unstack(["asset", "h"]).reindex(MODELS)
        c1 = R1[R1.target == t].groupby(["model", "asset", "h"])[met].mean().unstack(["asset", "h"]).reindex(MODELS)
        for m in MODELS:
            rows.append({"Pipeline": f"{t} {met}", "Model": m,
                         "Perubahan karena tuning (%)": (100 * (c0.loc[m] - c1.loc[m]) / c0.loc[m]).mean(),
                         "Tuning lebih baik di (sel)": f"{int((c1.loc[m] < c0.loc[m]).sum())}/{c0.shape[1]}"})
    display(Markdown("**Efek tuning pada setiap model**")); display(pd.DataFrame(rows).round(2))
    rows = []
    for t, met in [("return", "RMSE"), ("price", "RMSE"), ("price", "MAPE")]:
        for lab, R in (("tanpa tuning", R0), ("dengan tuning", R1)):
            cell = R[R.target == t].groupby(["model", "asset", "h"])[met].mean().unstack(["asset", "h"])
            for (c, sc_), name in CNAME.items():
                best = cell.loc[list(c)].min()
                rows.append({"Pipeline": f"{t} {met}", "Ensemble": name, "Versi": lab,
                             "Perbaikan vs anggota terbaik (%)": (100 * (best - cell.loc[name]) / best).mean(),
                             "Menang vs anggota terbaik": f"{int((cell.loc[name] < best).sum())}/{len(best)}"})
    CMP = pd.DataFrame(rows).round(2)
    display(Markdown("**Keunggulan ensemble atas anggota terbaiknya: tanpa vs dengan tuning**"))
    display(CMP.pivot_table(index=["Pipeline", "Ensemble"], columns="Versi",
                            values=["Perbaikan vs anggota terbaik (%)", "Menang vs anggota terbaik"], aggfunc="first", sort=False))
else:
    print("Belum bisa dibandingkan: jalankan notebook dengan TUNING = False dan TUNING = True (urutan bebas).")

## Catatan untuk penulisan
- Semua normalisasi dan parameter model di-fit hanya di data latih; walk-forward dengan jeda mencegah kebocoran data.
- Pada pipeline harga, jaringan saraf dengan min-max bisa kesulitan saat harga uji melampaui rentang data latih (mis. emas, BTC). Ini **bagian dari temuan**: model tunggal kalah di kondisi tertentu, dan ensemble diuji apakah bisa menutupinya.
- Prediksi tersimpan di `outputs/preds/` dengan nama yang sama seperti notebook sebelumnya, jadi kalau folder itu masih ada, training tidak perlu diulang (prediksi ARIMA/random walk di dalamnya diabaikan).
- Versi tanpa tuning: setelan sama untuk semua model. Versi dengan tuning: anggaran tuning sama untuk setiap model (10 percobaan, maks. 30 epoch), hanya memakai data sebelum tahun uji pertama.
- Keterbatasan: emas dan Brent adalah futures kontinu; horizon BTC dalam hari kalender.